# GENIE systematic covariances — Signal and Background separately (SB)

Compute per-knob **signal** and **background** covariance packs for GENIE MC,
separately rather than as a background-subtracted combination.

- **Signal**: GENIE uncertainty on signal (topo_categ == 1) events — both rate (reco-selected) and xsec (response-matrix path, no background term).
- **Background**: GENIE uncertainty on non-signal topology (topo_categ != 1) event rate.
- Outputs saved **parallel** to ``systematics-genie.ipynb`` with an ``SB`` infix in every path.
- Does **not** modify any existing framework module.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from os import path, makedirs
from datetime import datetime
from pathlib import Path
import json
import os
import gc
import pickle
import time
import warnings

import numpy as np
import pandas as pd
from pandas.errors import PerformanceWarning
from tqdm import tqdm

import sys
sys.path.append('/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana')

from pyanalib.split_df_helpers_new import dfs_from_dir
from pyanalib.covariance import get_covariance_matrix
from pyanalib.variable_calculator import (
    add_mc_cc1p0pi_tki_mcnu,
    add_reco_cc1p0pi_tki_evtdf,
    add_truth_cc1p0pi_tki_evtdf,
)

from analysis_village.numucc_1p0pi.variable_configs import VariableConfig
from analysis_village.numucc_1p0pi.categories import topology_list
from analysis_village.numucc_1p0pi.utils import (
    genie_univ_weight_series,
    get_clipped_evts,
    get_response_matrix,
    plot_frac_unc,
    plot_heatmap,
)
from analysis_village.numucc_1p0pi.files_config import save_fig_base_dir
from analysis_village.numucc_1p0pi.dataset_locations import (
    GENIE_GROUP_KNOBS,
    GENIE_GROUP_ORDER,
    _genie_glob_map,
)
from analysis_village.numucc_1p0pi.evt_derived_kinematics import ensure_derived_trk_kinematics_cols
from analysis_village.numucc_1p0pi.scripts.get_systematics_genie import (
    SystName,
    _align_evt_mcnu,
    _annotate_topo_genie_phi,
    _empty_xsec_tensor_acc,
    _xsec_cv_tensors,
    accumulate_xsec_path_chunk,
    genie_all_var_configs,
    genie_final_var_configs,
    normalize_and_infer_n_univ,
    sanitize_matrix_pack as _sanitize_matrix_pack,
    xsec_component_univ_events,
)
from analysis_village.numucc_1p0pi.syst_disk_layout import (
    FILE_GENIE,
    SUB_GENIE,
    category_out_dir,
    normalized_root,
)

warnings.filterwarnings('ignore', category=PerformanceWarning)
import matplotlib.pyplot as plt
plt.style.use('presentation.mplstyle')


def _ts():
    return datetime.now().strftime('%H:%M:%S')


def _log(msg):
    print(f'[{_ts()}] {msg}', flush=True)

In [ ]:
# --- run configuration (mirror of systematics-genie.ipynb) ---
INPUT_STAGE = os.environ.get('GENIE_MC_DF_STAGE', 'final')  # final | sel_all
_genie_groups_env = os.environ.get('GENIE_GROUPS', '')      # e.g. CCQE,MEC or empty = all
XSEC_UNIT = float(os.environ.get('GENIE_XSEC_UNIT', '1.0'))
MAX_FILES_PER_GROUP = int(os.environ.get('GENIE_MAX_FILES', '25'))

AR23_GROUPS = frozenset({'CCQE', 'MEC', 'RES', 'nonRES', 'DIS', 'Other'})

gmap = _genie_glob_map(INPUT_STAGE)
if _genie_groups_env.strip():
    GENIE_GROUPS = tuple(x.strip() for x in _genie_groups_env.split(',') if x.strip())
else:
    GENIE_GROUPS = tuple(g for g in GENIE_GROUP_ORDER if g in gmap)

GENIE_GROUPS = ('CCQE', 'MEC', 'RES', 'nonRES', 'DIS', 'Other', 'Ar23p')

bad = [g for g in GENIE_GROUPS if g not in gmap]
if bad:
    raise KeyError(f'unknown or empty GENIE group(s) {bad}; globs for stage={INPUT_STAGE!r}: {tuple(gmap)}')

var_configs = genie_all_var_configs(INPUT_STAGE) if INPUT_STAGE != 'final' else genie_final_var_configs()
_log(f'input_stage={INPUT_STAGE} groups={GENIE_GROUPS}')
_log(f'{len(var_configs)} variables; xsec_unit={XSEC_UNIT}')

In [ ]:
# --- data loading helpers (identical to systematics-genie.ipynb) ---

def genie_merged_search_dir(group: str) -> str:
    return str(Path(gmap[group]).parent)


def load_genie_group_frames(group: str):
    search_dir = genie_merged_search_dir(group)
    _log(f'loading {group} from {search_dir} ...')
    t0 = time.time()
    dfs = dfs_from_dir(
        search_dir=search_dir,
        filename_str='sel_mup',
        keys2load=['evt', 'mcnu'],
        n_max_concat=MAX_FILES_PER_GROUP,
        n_max_splits_per_file=None,
    )
    if 'evt' not in dfs or 'mcnu' not in dfs:
        raise RuntimeError(f'missing evt/mcnu for {group} under {search_dir}')
    evt, mcnu = _align_evt_mcnu(dfs['evt'], dfs['mcnu'])
    _log(f'  {group}: {len(evt):,} aligned evt rows in {time.time() - t0:.1f}s')
    return evt, mcnu


def prepare_genie_frames(evt: pd.DataFrame, mcnu: pd.DataFrame):
    evt = evt.copy()
    mcnu = mcnu.copy()
    evt = ensure_derived_trk_kinematics_cols(evt)
    evt = add_reco_cc1p0pi_tki_evtdf(evt)
    evt = add_truth_cc1p0pi_tki_evtdf(evt)
    mcnu = add_mc_cc1p0pi_tki_mcnu(mcnu)
    _annotate_topo_genie_phi(evt, mcnu)
    return _align_evt_mcnu(evt, mcnu)

In [ ]:
# --- output paths: parallel to systematics-genie.ipynb with 'SB' infix ---
today_str = 'integrated'
NOTEBOOK_OUT_ROOT = path.join(save_fig_base_dir, f'systematics-genie-SB-{today_str}')
makedirs(NOTEBOOK_OUT_ROOT, exist_ok=True)

SYST_DISK_ROOT = path.join(save_fig_base_dir, f'systematics-notebook-genie-SB-{today_str}')
genie_disk_dir = category_out_dir(SYST_DISK_ROOT, SUB_GENIE)
makedirs(genie_disk_dir, exist_ok=True)
COV_MAT_PKL = path.join(genie_disk_dir, FILE_GENIE)

_log('SB notebook NPZ root : ' + normalized_root(NOTEBOOK_OUT_ROOT))
_log('SB syst disk GENIE   : ' + normalized_root(genie_disk_dir))

SAVE_PER_GROUP_NPZ = True

In [ ]:
# =============================================================================
# SB-specific helpers
# None of the functions below modify any existing framework module.
# =============================================================================

def _init_cov_mat_dict_SB(var_configs: list) -> dict:
    """Initialise the SB covariance dict, parallel to _init_cov_mat_dict.

    Keys per variable:
      genie_signal          — total xsec (signal, R@N_gen path)
      genie_signal_rate     — total rate (signal, reco-selected)
      genie_bkgd_rate       — total rate (background topologies)
      genie_ar23_signal     — Ar23-group subset, signal xsec
      genie_ar23_signal_rate— Ar23-group subset, signal rate
      genie_ar23_bkgd_rate  — Ar23-group subset, background rate
    """
    out = {}
    for vc in var_configs:
        n = len(vc.bin_centers)
        z = np.zeros((n, n), dtype=np.float64)
        out[vc.var_save_name] = {
            'genie_signal':           z.copy(),
            'genie_signal_rate':      z.copy(),
            'genie_bkgd_rate':        z.copy(),
            'genie_ar23_signal':      z.copy(),
            'genie_ar23_signal_rate': z.copy(),
            'genie_ar23_bkgd_rate':   z.copy(),
        }
    return out


def _empty_sb_rate_acc(n_univ: int, nb: int) -> dict:
    """Accumulator for reco-selected **signal** rate universes (topo_categ==1).

    Background rate universes are read directly from the xsec accumulator
    (acc_xsec['bg_cv'] / acc_xsec['bg_univ']) which uses the same reco variable.
    """
    return {
        'signal_rate_cv':   np.zeros(nb,           dtype=np.float64),
        'signal_rate_univ': np.zeros((n_univ, nb), dtype=np.float64),
    }


def accumulate_sb_signal_rate_chunk(
    mc_evt_df: pd.DataFrame,
    mc_nu_df: pd.DataFrame,
    var_config,
    syst_name: SystName,
    n_univ: int,
    acc_rate: dict,
) -> None:
    """Fill acc_rate with per-universe signal-rate histograms (reco variable, topo_categ==1)."""
    bins = var_config.bins
    evtdf_signal = mc_evt_df[mc_evt_df.topo_categ == 1]

    var_sig, wgt_sig = get_clipped_evts(
        evtdf_signal,
        var_config.var_evt_reco_col,
        bins,
        var_save_name=var_config.var_save_name,
    )
    acc_rate['signal_rate_cv'] += np.histogram(var_sig, bins=bins, weights=wgt_sig)[0].astype(np.float64)

    wblock = evtdf_signal[syst_name]
    for uidx in range(n_univ):
        w = genie_univ_weight_series(wblock, uidx)
        acc_rate['signal_rate_univ'][uidx] += np.histogram(
            var_sig, bins=bins, weights=wgt_sig * w
        )[0].astype(np.float64)


def build_sb_packs(acc_xsec: dict, acc_rate: dict, xsec_unit: float) -> dict:
    """Compute signal and background covariance packs from the two accumulators.

    Returns a dict with three keys, each a ``{cov, cov_frac, corr}`` pack:

    ``signal_rate``
        Fractional covariance of reco-selected signal events (topo_categ==1)
        across GENIE universes.
    ``bkgd_rate``
        Fractional covariance of reco-selected background events (non-signal
        topologies) across GENIE universes.  Taken from acc_xsec['bg_*'] which
        is accumulated identically to get_univ_rates background loop.
    ``signal_xsec``
        Fractional covariance of the signal cross-section via the response-matrix
        path:  R(eff_u, reco_u) @ N_gen^CV.  No background term.
    """
    scale = float(xsec_unit) if xsec_unit else 1.0

    # background rate: reco-space non-signal topology events
    bkgd_cv   = np.asarray(acc_xsec['bg_cv'],   dtype=np.float64)
    bkgd_univ = np.asarray(acc_xsec['bg_univ'], dtype=np.float64)
    pack_bkgd_rate = _sanitize_matrix_pack(
        get_covariance_matrix(bkgd_univ, bkgd_cv)
    )

    # signal rate: reco-space signal events
    sig_rate_cv   = np.asarray(acc_rate['signal_rate_cv'],   dtype=np.float64)
    sig_rate_univ = np.asarray(acc_rate['signal_rate_univ'], dtype=np.float64)
    pack_sig_rate = _sanitize_matrix_pack(
        get_covariance_matrix(sig_rate_univ, sig_rate_cv)
    )

    # signal xsec: R(eff_u, reco_u) @ N_gen^CV, no background correction
    sig_xsec_univ = xsec_component_univ_events(acc_xsec, scale, 'signal')
    eff_cv, reco_cv, _ = _xsec_cv_tensors(acc_xsec)
    sig_xsec_cv = (
        get_response_matrix(reco_cv, eff_cv)
        @ np.asarray(acc_xsec['nevts_allmc'], dtype=np.float64)
        * scale
    )
    pack_sig_xsec = _sanitize_matrix_pack(
        get_covariance_matrix(sig_xsec_univ, sig_xsec_cv)
    )

    return {
        'signal_rate': pack_sig_rate,
        'bkgd_rate':   pack_bkgd_rate,
        'signal_xsec': pack_sig_xsec,
    }


def syst_dict_var_first_SB(group_syst_by_var: dict) -> dict:
    return {slug: np.array(by_knob, dtype=object) for slug, by_knob in group_syst_by_var.items()}


def run_group_in_memory_SB(group: str, cov_mat_dict_SB: dict):
    """Signal/Background covariance production for one GENIE group.

    Parallel to run_group_in_memory from systematics-genie.ipynb, but writes
    signal_rate / bkgd_rate / signal_xsec packs instead of combined rate/xsec.
    """
    knobs = list(GENIE_GROUP_KNOBS.get(group, ()))
    if not knobs:
        _log(f'SKIP {group}: no knobs in GENIE_GROUP_KNOBS')
        return None

    evt, mcnu = load_genie_group_frames(group)
    mc_evt, mc_nu = prepare_genie_frames(evt, mcnu)
    del evt, mcnu
    gc.collect()

    group_syst_by_var = {vc.var_save_name: {} for vc in var_configs}
    t_group = time.time()

    for iknob, knob in enumerate(tqdm(knobs, desc=f'knobs ({group})')):
        syst_name: SystName = ('mc', knob)
        try:
            n_univ = normalize_and_infer_n_univ(mc_evt, mc_nu, syst_name)
        except Exception as ex:
            _log(f'  SKIP {group}/{knob} (n_univ): {ex}')
            continue

        for ivar, vc in enumerate(var_configs):
            slug = vc.var_save_name
            nb = len(vc.bins) - 1
            if iknob == 0:
                _log(f'--- {group} variable [{ivar + 1}/{len(var_configs)}] {slug} ---')
            try:
                # xsec accumulator: provides signal xsec tensors + background rate
                acc_xsec = _empty_xsec_tensor_acc(n_univ, nb)
                accumulate_xsec_path_chunk(mc_evt, mc_nu, vc, syst_name, n_univ, acc_xsec)

                # rate accumulator: reco-selected signal events per universe
                acc_rate = _empty_sb_rate_acc(n_univ, nb)
                accumulate_sb_signal_rate_chunk(mc_evt, mc_nu, vc, syst_name, n_univ, acc_rate)

                packs = build_sb_packs(acc_xsec, acc_rate, XSEC_UNIT)
            except Exception as ex:
                _log(f'  SKIP {group}/{knob}/{slug}: {ex}')
                continue

            group_syst_by_var[slug][knob] = packs

            row = cov_mat_dict_SB.get(slug)
            if row is None:
                continue

            # signal rate
            rk = f'{knob}_signal_rate'
            if rk not in row:
                row[rk] = np.zeros_like(packs['signal_rate']['cov_frac'])
            row[rk] += packs['signal_rate']['cov_frac']
            row['genie_signal_rate'] += packs['signal_rate']['cov_frac']
            if group in AR23_GROUPS:
                row['genie_ar23_signal_rate'] += packs['signal_rate']['cov_frac']

            # background rate
            bk = f'{knob}_bkgd_rate'
            if bk not in row:
                row[bk] = np.zeros_like(packs['bkgd_rate']['cov_frac'])
            row[bk] += packs['bkgd_rate']['cov_frac']
            row['genie_bkgd_rate'] += packs['bkgd_rate']['cov_frac']
            if group in AR23_GROUPS:
                row['genie_ar23_bkgd_rate'] += packs['bkgd_rate']['cov_frac']

            # signal xsec
            xk = f'{knob}_signal'
            if xk not in row:
                row[xk] = np.zeros_like(packs['signal_xsec']['cov_frac'])
            row[xk] += packs['signal_xsec']['cov_frac']
            row['genie_signal'] += packs['signal_xsec']['cov_frac']
            if group in AR23_GROUPS:
                row['genie_ar23_signal'] += packs['signal_xsec']['cov_frac']

    group_syst_by_var = {s: d for s, d in group_syst_by_var.items() if d}
    _log(f'FINISHED {group} in {time.time() - t_group:.1f}s')

    if SAVE_PER_GROUP_NPZ and group_syst_by_var:
        grp_dir = path.join(NOTEBOOK_OUT_ROOT, f'systematics-genie-SB-{group}-{today_str}')
        makedirs(grp_dir, exist_ok=True)
        npz_path = path.join(grp_dir, f'genie-SB-{group}_syst_dict.npz')
        np.savez_compressed(npz_path, **syst_dict_var_first_SB(group_syst_by_var))
        _log(f'  wrote {npz_path}')

    del mc_evt, mc_nu
    gc.collect()
    return group_syst_by_var

In [ ]:
# Limit variables (mirror of systematics-genie.ipynb Cell 7)
var_configs = [VariableConfig.all_events(),
                VariableConfig.muon_momentum(),
                VariableConfig.muon_direction(),
                VariableConfig.proton_momentum(),
                VariableConfig.proton_direction(),
                VariableConfig.tki_del_alpha(),
                VariableConfig.tki_del_phi(),
                VariableConfig.tki_del_Tp(),
                VariableConfig.tki_del_p(),
                VariableConfig.tki_del_Tp_x(),
                VariableConfig.tki_del_Tp_y(),
                # VariableConfig.muon_direction_x(),
                # VariableConfig.muon_direction_y(),
                # VariableConfig.proton_direction_x(),
                # VariableConfig.proton_direction_y(),
                # # VariableConfig.opening_angle(),
                # VariableConfig.vertex_x(),
                # VariableConfig.vertex_y(),
                # VariableConfig.vertex_z(),
                ]

In [ ]:
# --- main production: Signal / Background separately ---
cov_mat_dict_SB = _init_cov_mat_dict_SB(var_configs)

t_all = time.time()
for group in GENIE_GROUPS:
    _log(f'=== GENIE group {group} ===')
    run_group_in_memory_SB(group, cov_mat_dict_SB)

with open(COV_MAT_PKL, 'wb') as f:
    pickle.dump(cov_mat_dict_SB, f, protocol=pickle.HIGHEST_PROTOCOL)
_log(f'wrote {COV_MAT_PKL} in {time.time() - t_all:.1f}s')

manifest = {
    'schema': 'numucc_genie_cov_mat_dict_SB_v1',
    'description': (
        'GENIE covariance for signal and background separately '
        '(not background-subtracted). '
        'Parallel output to systematics-notebook-genie-<date>.'
    ),
    'mc_df_stage': INPUT_STAGE,
    'genie_groups': list(GENIE_GROUPS),
    'variables': sorted(cov_mat_dict_SB.keys()),
    'xsec_unit': XSEC_UNIT,
    'output_pkl': COV_MAT_PKL,
    'per_group_npz_root': NOTEBOOK_OUT_ROOT,
    'keys_per_variable': [
        'genie_signal        -- total signal xsec (R@N_gen path, Ar23+Ar23p)',
        'genie_signal_rate   -- total signal rate (reco-selected, topo==1)',
        'genie_bkgd_rate     -- total background rate (non-signal topos)',
        'genie_ar23_signal   -- Ar23 groups only, signal xsec',
        'genie_ar23_signal_rate -- Ar23 groups only, signal rate',
        'genie_ar23_bkgd_rate   -- Ar23 groups only, background rate',
        '+ per-knob: {knob}_signal, {knob}_signal_rate, {knob}_bkgd_rate',
    ],
}
manifest_path = path.join(genie_disk_dir, 'genie_SB_covariance_manifest.json')
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)
_log('wrote ' + manifest_path)

In [ ]:
# --- Quick summary: integrated variable (single-bin) ---
if 'integrated' in cov_mat_dict_SB:
    row = cov_mat_dict_SB['integrated']
    print('Integrated (single-bin) GENIE fractional uncertainties [%]')
    print('-' * 55)
    label_keys = [
        ('genie_signal_rate',      'Signal rate          (Ar23+)'),
        ('genie_bkgd_rate',        'Background rate      (Ar23+)'),
        ('genie_signal',           'Signal xsec          (Ar23+)'),
        ('genie_ar23_signal_rate', 'Signal rate          (Ar23 only)'),
        ('genie_ar23_bkgd_rate',   'Background rate      (Ar23 only)'),
        ('genie_ar23_signal',      'Signal xsec          (Ar23 only)'),
    ]
    for key, label in label_keys:
        val = row.get(key)
        if val is not None:
            unc = 100.0 * np.sqrt(max(float(np.asarray(val).flat[0]), 0.0))
            print(f'  {label:35s}:  {unc:.4f}%')
else:
    _log('integrated variable not found in cov_mat_dict_SB — did the run complete?')

## Diagnostics: NormCCMEC, multisigma / morph, and multiplied-universe GENIE total

### A. NormCCMEC vs true CC MEC

`GENIEReWeight_SBN_v1_multisim_NormCCMEC` scales the CC MEC model. On **true CC MEC** events (`get_genie_category == 2`) the multisim spread in the integrated weighted rate should be larger than on the **inclusive** sample; the ratio of fractional uncertainties should track the CV weighted fraction of MEC in the sample (often order **½** when that fraction is ~50%).

### B. Multisigma and morph (example knobs)

The production path uses **one** synthetic universe for multisigma (`ps1` only) and morph (`morph` only); see `normalize_and_infer_n_univ` and `genie_univ_weight_series`. The code cell compares **integrated weighted event rates** to (i) for multisigma: the **mean of the absolute fractional shifts** from `ps1` and `ms1`; (ii) for morph: the **absolute fractional shift** from the morph column (first pure-morph knob found in the configured groups, if any).

### C. Flux-style multiplied GENIE + Gaussian non-multisim knobs

The next cell builds **multiplied** per-universe weights across all knobs (multisim from file), turns multisigma/morph into **Gaussian** pseudo-universes (recipes aligned with `makedf/getsyst.py` **slim** mode, with **separate RNG seeds** for multisigma vs morph), and compares spread of the total rate to the **additive** per-knob fractional covariance already stored in `cov_mat_dict_SB`. It also forms **per GENIE interaction mode** spreads and reports `sqrt(sum σ_m²)`.


In [ ]:
# --- NormCCMEC (true CC MEC) + multisigma / morph rate diagnostics ---
# Requires `cov_mat_dict_SB` from the production run and the same `load_genie_group_frames` / `prepare_genie_frames`.

from analysis_village.numucc_1p0pi.categories import get_genie_category

# Optional: subsample events for the heavy multiplied-universe cell later (this cell uses full MEC / RES loads).
DIAG_MAX_FILES = int(os.environ.get('GENIE_DIAG_MAX_FILES', str(MAX_FILES_PER_GROUP)))


def _weight_leaf(col) -> str:
    if not isinstance(col, tuple):
        return str(col)
    for x in col:
        if x != '' and x is not None:
            return str(x)
    return str(col[0])


def _genie_block(df: pd.DataFrame, knob: str) -> pd.DataFrame:
    return df.loc[:, ('mc', knob)]


def _series_by_leaf(block: pd.DataFrame, leaf: str) -> pd.Series:
    for c in block.columns:
        if _weight_leaf(c) == leaf:
            return block[c]
    raise KeyError(f'leaf {leaf!r} not in knob block; have {sorted({_weight_leaf(c) for c in block.columns})}')


def _infer_n_univ(block: pd.DataFrame) -> int:
    m = -1
    for c in block.columns:
        s = _weight_leaf(c)
        if s.startswith('univ_'):
            try:
                m = max(m, int(s.split('_', 1)[1]))
            except ValueError:
                pass
    return m + 1


def _pot_weight_series(evt: pd.DataFrame) -> pd.Series:
    """Per-event POT scale — same rules as ``get_clipped_evts`` in ``utils``.

    GENIE ``.df`` bundles often use a **MultiIndex**; ``'pot_weight' in df.columns`` is
    then false even when a ``pot_weight`` leaf exists, and histograms use unit weights.
    """
    if 'pot_weight' in evt.columns:
        return evt['pot_weight'].astype(np.float64)
    if isinstance(evt.columns, pd.MultiIndex):
        hits = [c for c in evt.columns if _weight_leaf(c) == 'pot_weight']
        if hits:
            return evt.loc[:, hits[0]].astype(np.float64)
    return pd.Series(1.0, index=evt.index, dtype=np.float64)


def _integrated_weighted_rate(evt: pd.DataFrame, mask: pd.Series, w_extra: pd.Series | None = None) -> float:
    """Sum pot_weight (optionally times per-event w_extra) over mask."""
    base = _pot_weight_series(evt).loc[mask].astype(np.float64)
    if w_extra is None:
        return float(base.sum())
    return float((base * w_extra.loc[mask].astype(np.float64)).sum())


def _frac_unc_from_univ_hist(cv: float, univ: np.ndarray) -> float:
    """sqrt(diag) fractional std from get_covariance_matrix (same as notebook)."""
    cv = np.asarray([cv], dtype=np.float64)
    u = np.asarray(univ, dtype=np.float64).reshape(1, -1)
    pack = get_covariance_matrix(u, cv)
    return float(np.sqrt(max(pack['cov_frac'][0, 0], 0.0)))


# ----- 1) NormCCMEC: inclusive vs true CC MEC (genie category == 2) -----
KNOB_NORM_CCMEC = 'GENIEReWeight_SBN_v1_multisim_NormCCMEC'
_log(f'NormCCMEC diagnostic: loading MEC group (max_files={DIAG_MAX_FILES}) ...')
_prev_max = MAX_FILES_PER_GROUP
try:
    globals()['MAX_FILES_PER_GROUP'] = DIAG_MAX_FILES
    evt_mec, mcnu_mec = load_genie_group_frames('MEC')
finally:
    globals()['MAX_FILES_PER_GROUP'] = _prev_max

mc_evt_mec, _ = prepare_genie_frames(evt_mec, mcnu_mec)
del evt_mec, mcnu_mec
gc.collect()

blk = _genie_block(mc_evt_mec, KNOB_NORM_CCMEC)
n_univ_ccmec = _infer_n_univ(blk)
if n_univ_ccmec <= 0:
    _log('NormCCMEC: no multisim univ_* columns; skip')
else:
    genie_cat = get_genie_category(mc_evt_mec)
    mask_mec_truth = genie_cat == 2  # FV numu CC MEC
    mask_all = pd.Series(True, index=mc_evt_mec.index)

    cv_all = _integrated_weighted_rate(mc_evt_mec, mask_all)
    cv_mt = _integrated_weighted_rate(mc_evt_mec, mask_mec_truth)
    f_mec = cv_mt / max(cv_all, 1e-12)

    univ_all = np.array(
        [_integrated_weighted_rate(mc_evt_mec, mask_all, _series_by_leaf(blk, f'univ_{u}')) for u in range(n_univ_ccmec)]
    )
    univ_mt = np.array(
        [_integrated_weighted_rate(mc_evt_mec, mask_mec_truth, _series_by_leaf(blk, f'univ_{u}')) for u in range(n_univ_ccmec)]
    )

    sig_all = _frac_unc_from_univ_hist(cv_all, univ_all)
    sig_mt = _frac_unc_from_univ_hist(cv_mt, univ_mt)

    print('--- NormCCMEC (multisim) integrated weighted rate ---')
    print(f'  n_univ={n_univ_ccmec}, N_events={len(mc_evt_mec):,}, N_true_CC_MEC={int(mask_mec_truth.sum()):,}')
    print(f'  weighted MEC fraction (CV) R_mec/R_tot = {100*f_mec:.3f}%')
    print(f'  fractional unc inclusive: {100*sig_all:.4f}%')
    print(f'  fractional unc true CC MEC only: {100*sig_mt:.4f}%')
    print(f'  ratio inclusive / MEC-truth = {sig_all / max(sig_mt, 1e-12):.4f}')
    print(
        '  (For a knob that multiplies only the MEC cross section, expect σ_inclusive ≈ '
        '(R_mec/R_tot) × σ_MEC-truth if the relative weight shift is shared by all MEC events.)'
    )
    print(f'  predicted ratio from fractions ≈ {f_mec:.4f} (compare to line above)')

del mc_evt_mec
gc.collect()

# ----- 2) Multisigma example: MaCCRES (ps1 / ms1) vs pipeline (ps1-only universe) -----
EX_MULTISIGMA = 'GENIEReWeight_SBN_v1_multisigma_MaCCRES'
_log(f'Multisigma diagnostic: loading RES group for {EX_MULTISIGMA} ...')
_prev_max = MAX_FILES_PER_GROUP
try:
    globals()['MAX_FILES_PER_GROUP'] = DIAG_MAX_FILES
    evt_res, mcnu_res = load_genie_group_frames('RES')
finally:
    globals()['MAX_FILES_PER_GROUP'] = _prev_max

mc_evt_res, _ = prepare_genie_frames(evt_res, mcnu_res)
del evt_res, mcnu_res
gc.collect()

blk_ms = _genie_block(mc_evt_res, EX_MULTISIGMA)
w_ps1 = _series_by_leaf(blk_ms, 'ps1')
w_ms1 = _series_by_leaf(blk_ms, 'ms1')

for label, mask in [('all selected', pd.Series(True, index=mc_evt_res.index)), ('background topo !=1', mc_evt_res.topo_categ != 1)]:
    cv = _integrated_weighted_rate(mc_evt_res, mask)
    r_ps = _integrated_weighted_rate(mc_evt_res, mask, w_ps1)
    r_ms = _integrated_weighted_rate(mc_evt_res, mask, w_ms1)
    dps = (r_ps / cv) - 1.0
    dms = (r_ms / cv) - 1.0
    avg_two_sided = 0.5 * (abs(dps) + abs(dms))
    r_pipe = np.array([_integrated_weighted_rate(mc_evt_res, mask, w_ps1)])
    sig_pipe = _frac_unc_from_univ_hist(cv, r_pipe)
    print(f'--- Multisigma {EX_MULTISIGMA} — {label} ---')
    print(f'  CV rate={cv:.6g},  frac shift ps1={100*dps:.4f}%, ms1={100*dms:.4f}%')
    print(f'  mean(|d_ps1|,|d_ms1|) = {100*avg_two_sided:.4f}%  (symmetric two-sided scale)')
    print(f'  pipeline sqrt(cov_frac) using ps1-only synthetic universe = {100*sig_pipe:.4f}%')
    print(f'  ratio pipeline / mean-two-sided = {sig_pipe / max(avg_two_sided, 1e-12):.4f}')

del mc_evt_res
gc.collect()

# ----- 3) Morph example: first knob in any group with a morph leaf (unisim type-3, no univ_*) -----
EX_MORPH_KNOB = None
EX_MORPH_GROUP = None
mc_evt_morph = None
_prev_max = MAX_FILES_PER_GROUP
try:
    globals()['MAX_FILES_PER_GROUP'] = DIAG_MAX_FILES
    for grp in GENIE_GROUPS:
        evt_x, mcnu_x = load_genie_group_frames(grp)
        mc_x, _ = prepare_genie_frames(evt_x, mcnu_x)
        del evt_x, mcnu_x
        for cand in GENIE_GROUP_KNOBS.get(grp, ()):
            b = _genie_block(mc_x, cand)
            leaves = {_weight_leaf(c) for c in b.columns}
            if 'morph' in leaves and not any(x.startswith('univ_') for x in leaves):
                EX_MORPH_KNOB = cand
                EX_MORPH_GROUP = grp
                mc_evt_morph = mc_x
                break
        if EX_MORPH_KNOB is not None:
            break
        del mc_x
        gc.collect()
finally:
    globals()['MAX_FILES_PER_GROUP'] = _prev_max

if EX_MORPH_KNOB is None:
    print(
        '--- Morph diagnostic: no pure morph-only knob found in loaded GENIE groups '
        '(many SBN knobs are multisim or multisigma/ps1). Skip morph-vs-pipeline check. ---'
    )
else:
    blk_m = _genie_block(mc_evt_morph, EX_MORPH_KNOB)
    w_morph = _series_by_leaf(blk_m, 'morph')
    for label, mask in [('all selected', pd.Series(True, index=mc_evt_morph.index)), ('background topo !=1', mc_evt_morph.topo_categ != 1)]:
        cv = _integrated_weighted_rate(mc_evt_morph, mask)
        r_m = _integrated_weighted_rate(mc_evt_morph, mask, w_morph)
        dm = abs((r_m / cv) - 1.0)
        r_pipe = np.array([_integrated_weighted_rate(mc_evt_morph, mask, w_morph)])
        sig_pipe = _frac_unc_from_univ_hist(cv, r_pipe)
        print(f'--- Morph knob {EX_MORPH_KNOB} ({EX_MORPH_GROUP}) — {label} ---')
        print(f'  |frac rate shift morph| = {100*dm:.4f}%')
        print(f'  pipeline sqrt(cov_frac) morph-only universe = {100*sig_pipe:.4f}%')
        print(f'  ratio pipeline / |morph| = {sig_pipe / max(dm, 1e-12):.4f}')
    del mc_evt_morph
    gc.collect()


### Multiplied GENIE “Flux total” style + Gaussian knobs

**Model:** For universe index `i`, each event gets the **product** of all GENIE knob weights at that universe (same indexing idea as `systematics.ipynb` / Flux: one combined draw per universe). **Multisim** knobs use stored `univ_*` weights (`i mod n_univ` per knob if lengths differ). **Multisigma** knobs are turned into pseudo-universes with draws `z ~ N(0,1)` and per-event weights `1 + z * (ps1 - 1)`, matching the **slim** recipe in `makedf/getsyst.py` (not the same as using `ps1` alone in the covariance path). **Morph** knobs use `1 + (morph - 1) * 2 * |z|` with independent `z`, also matching that slim block.

**RNG:** `numpy.random.Generator` instances with **fixed, distinct seeds** feed multisigma vs morph Gaussian draws so those dimensions are statistically independent of each other and of the file-based multisim weights.

**Interaction modes:** `get_genie_category` codes (1=CCQE, 2=MEC, 3=RES, 5=Other CC, 6=NC, 0=other ν, −1=cosmic). For each mode `m`, take the fractional spread `σ_m = std_i(R_m^{(i)})/R_m^{\\mathrm{CV}}` with `R_m^{(i)} = \\sum_{e\\in m} \\mathrm{pot\\_weight}_e \\prod_k w_{k}^{(i)}(e)`. Report `\\sqrt{\\sum_m \\sigma_m^2}` and compare to the **diagonal** fractional uncertainties already accumulated in `cov_mat_dict_SB['integrated']` (`genie_signal_rate`, `genie_bkgd_rate` — independent sum of per-knob fractional covariances, not the same construction as the product model).


In [ ]:
# --- Multiplied GENIE universes (Flux-style) + Gaussian multisigma/morph ---
# Re-loads each GENIE group and inner-joins all knob columns on event index (same MC sample).

N_UNIV_PRODUCT = int(os.environ.get('GENIE_DIAG_PRODUCT_N_UNIV', '250'))
SEED_MULTISIG_GAUSS = int(os.environ.get('GENIE_DIAG_SEED_MS', '1906026026'))
SEED_MORPH_GAUSS = int(os.environ.get('GENIE_DIAG_SEED_MORPH', '2906026026'))

_prev_max = MAX_FILES_PER_GROUP
try:
    globals()['MAX_FILES_PER_GROUP'] = DIAG_MAX_FILES
    g0 = GENIE_GROUPS[0]
    evt0, mcnu0 = load_genie_group_frames(g0)
    merged, _ = prepare_genie_frames(evt0, mcnu0)
    del evt0, mcnu0
    gc.collect()

    for grp in GENIE_GROUPS:
        evt_g, mcnu_g = load_genie_group_frames(grp)
        mc_g, _ = prepare_genie_frames(evt_g, mcnu_g)
        del evt_g, mcnu_g
        for knob in GENIE_GROUP_KNOBS.get(grp, ()):
            colkey = ('mc', knob)
            if colkey not in mc_g.columns:
                continue
            if colkey not in merged.columns:
                merged = merged.join(mc_g[[colkey]], how='inner')
        del mc_g
        gc.collect()
finally:
    globals()['MAX_FILES_PER_GROUP'] = _prev_max

pot = _pot_weight_series(merged).astype(np.float64).values
genie_cat = get_genie_category(merged).values
topo = merged['topo_categ'].values

# --- classify knobs present on merged frame ---
multisim_mats = []  # list of (name, n_univ, W n_events x n_univ)
ms_ps1 = []  # list of (name, ps1 vector)
morph_vecs = []  # list of (name, morph vector)

for grp in GENIE_GROUPS:
    for knob in GENIE_GROUP_KNOBS.get(grp, ()):
        colkey = ('mc', knob)
        if colkey not in merged.columns:
            continue
        blk = merged[colkey]
        leaves = {_weight_leaf(c) for c in blk.columns}
        n_u = _infer_n_univ(blk)
        if n_u > 1:
            cols = [np.nan_to_num(_series_by_leaf(blk, f'univ_{u}').values.astype(np.float64), nan=1.0, posinf=1.0, neginf=1.0) for u in range(n_u)]
            W = np.column_stack(cols)
            multisim_mats.append((knob, n_u, W))
        elif 'ps1' in leaves:
            ps1 = np.nan_to_num(_series_by_leaf(blk, 'ps1').values.astype(np.float64), nan=1.0, posinf=1.0, neginf=1.0)
            ms_ps1.append((knob, ps1))
        elif 'morph' in leaves:
            morph = np.nan_to_num(_series_by_leaf(blk, 'morph').values.astype(np.float64), nan=1.0, posinf=1.0, neginf=1.0)
            morph_vecs.append((knob, morph))

n_ev = len(merged)
_log(
    f'merged GENIE weight table: {n_ev:,} events, '
    f'{len(multisim_mats)} multisim knobs, {len(ms_ps1)} multisigma, {len(morph_vecs)} morph'
)

rng_ms = np.random.default_rng(SEED_MULTISIG_GAUSS)
rng_mo = np.random.default_rng(SEED_MORPH_GAUSS)
Z_ms = rng_ms.standard_normal((N_UNIV_PRODUCT, len(ms_ps1)))
Z_mo = rng_mo.standard_normal((N_UNIV_PRODUCT, len(morph_vecs)))

rates_all = np.zeros(N_UNIV_PRODUCT)
rates_sig = np.zeros(N_UNIV_PRODUCT)
rates_bkg = np.zeros(N_UNIV_PRODUCT)

mode_ids = np.array([1, 2, 3, 5, 6, 0, -1], dtype=np.int64)
rates_mode = np.zeros((N_UNIV_PRODUCT, len(mode_ids)))
cv_mode = np.zeros(len(mode_ids))

for mi, mid in enumerate(mode_ids):
    m = genie_cat == mid
    cv_mode[mi] = float(pot[m].sum())

cv_all = float(pot.sum())
cv_sig = float(pot[topo == 1].sum())
cv_bkg = float(pot[topo != 1].sum())

for i in range(N_UNIV_PRODUCT):
    wtot = np.ones(n_ev, dtype=np.float64)
    for name, n_u, W in multisim_mats:
        wtot *= W[:, i % n_u]
    for j, (_, ps1) in enumerate(ms_ps1):
        wtot *= np.clip(1.0 + Z_ms[i, j] * (ps1 - 1.0), 0.0, np.inf)
    for j, (_, mv) in enumerate(morph_vecs):
        wtot *= np.maximum(1.0 + (mv - 1.0) * 2.0 * np.abs(Z_mo[i, j]), 0.0)

    rates_all[i] = float((pot * wtot).sum())
    rates_sig[i] = float((pot[topo == 1] * wtot[topo == 1]).sum())
    rates_bkg[i] = float((pot[topo != 1] * wtot[topo != 1]).sum())
    for mi, mid in enumerate(mode_ids):
        m = genie_cat == mid
        rates_mode[i, mi] = float((pot[m] * wtot[m]).sum())


def _rel_std(cv: float, rs: np.ndarray) -> float:
    if cv <= 0:
        return float('nan')
    x = rs / cv - 1.0
    return float(np.std(x, ddof=0))


sig_mul_all = _rel_std(cv_all, rates_all)
sig_mul_sig = _rel_std(cv_sig, rates_sig)
sig_mul_bkg = _rel_std(cv_bkg, rates_bkg)

sig_mode_quad = 0.0
mode_fracs = []
for mi, mid in enumerate(mode_ids):
    cv = cv_mode[mi]
    if cv <= 0:
        continue
    sm = _rel_std(cv, rates_mode[:, mi])
    mode_fracs.append((int(mid), sm))
    sig_mode_quad += sm * sm
sig_mode_quad = float(np.sqrt(max(sig_mode_quad, 0.0)))

print('--- Multiplied GENIE + Gaussian (non-multisim) — integrated weighted rates ---')
print(f'  N_universes={N_UNIV_PRODUCT}, independent Z_ms (seed={SEED_MULTISIG_GAUSS}), Z_morph (seed={SEED_MORPH_GAUSS})')
print(f'  fractional std (multiply model) all events: {100*sig_mul_all:.4f}%')
print(f'  fractional std (multiply model) signal topo==1: {100*sig_mul_sig:.4f}%')
print(f'  fractional std (multiply model) background topo!=1: {100*sig_mul_bkg:.4f}%')
print('  per GENIE mode fractional std (same combined wtot):')
for mid, sm in sorted(mode_fracs, key=lambda t: -t[1]):
    print(f'    mode {mid:3d}: {100*sm:.4f}%')
print(f'  quadrature sum of mode stds sqrt(sum σ_m^2): {100*sig_mode_quad:.4f}%')

if 'integrated' in cov_mat_dict_SB:
    row = cov_mat_dict_SB['integrated']

    def _sqrt_diag(key):
        a = row.get(key)
        if a is None:
            return float('nan')
        return float(np.sqrt(max(float(np.asarray(a).flat[0]), 0.0)))

    add_sig = _sqrt_diag('genie_signal_rate')
    add_bkg = _sqrt_diag('genie_bkgd_rate')
    add_indep = float(np.sqrt(max(add_sig ** 2 + add_bkg ** 2, 0.0)))  # not cross-correlated across topo
    print('--- Compare to additive per-knob covariance (diagonal, integrated) ---')
    print(f'  sqrt(diag genie_signal_rate) additive: {100*add_sig:.4f}%')
    print(f'  sqrt(diag genie_bkgd_rate)   additive: {100*add_bkg:.4f}%')
    print(f'  naive quadrature signal⊕bkg:        {100*add_indep:.4f}%  (ignores topo correlation)')
    print(f'  multiply-model / additive-signal:   {sig_mul_sig / max(add_sig, 1e-12):.4f}')
    print(f'  multiply-model / additive-bkg:      {sig_mul_bkg / max(add_bkg, 1e-12):.4f}')
    print(f'  multiply-all / naive-signal⊕bkg:     {sig_mul_all / max(add_indep, 1e-12):.4f}')
    print(f'  mode-quadrature / multiply-all:     {sig_mode_quad / max(sig_mul_all, 1e-12):.4f}')
else:
    _log('cov_mat_dict_SB missing integrated — skip additive comparison')

del merged
gc.collect()


In [ ]:
# --- Compare to systematics-genie.ipynb totals (optional) ---
# Load the original pickle to compare signal+bkgd uncertainties
COMPARE_ORIGINAL = False
ORIGINAL_PKL = path.join(
    category_out_dir(
        path.join(save_fig_base_dir, f'systematics-notebook-genie-{today_str}'),
        SUB_GENIE,
    ),
    FILE_GENIE,
)
if COMPARE_ORIGINAL and path.isfile(ORIGINAL_PKL):
    with open(ORIGINAL_PKL, 'rb') as f:
        orig = pickle.load(f)
    if 'integrated' in orig and 'integrated' in cov_mat_dict_SB:
        o = orig['integrated']
        s = cov_mat_dict_SB['integrated']
        print('--- Comparison: original (bkgd-subtracted) vs SB ---')
        pairs = [
            ('genie_rate',    'original rate (subtracted)',     'genie_signal_rate', 'SB signal rate'),
            ('genie',         'original xsec (subtracted)',     'genie_signal',      'SB signal xsec'),
        ]
        for ok, olabel, sk, slabel in pairs:
            ov = o.get(ok)
            sv = s.get(sk)
            if ov is not None and sv is not None:
                print(f'  {olabel:38s}: {100*np.sqrt(max(float(ov.flat[0]),0)):.4f}%')
                print(f'  {slabel:38s}: {100*np.sqrt(max(float(sv.flat[0]),0)):.4f}%')
                print()
else:
    _log(f'COMPARE_ORIGINAL=False or original pickle not found: {ORIGINAL_PKL}')